In [1]:
# Imports

import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [2]:
# Generate Synthetic Data
fake = Faker()
random.seed(42)
np.random.seed(42)

In [3]:
# Settings
NUM_ACCOUNTS = 50
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META', 'NVDA', 'JPM']
METHODS = ['FIFO', 'FIFO', 'FIFO', 'LIFO', 'SPEC_ID']  # FIFO weighted higher

def random_date(start_year=2020, end_year=2024):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    return start + timedelta(days=random.randint(0, (end - start).days))

In [4]:
# Generate transactions
transactions = []
for i in range(500):
    qty = random.randint(1, 100)
    price = round(random.uniform(50, 800), 2)
    
    # Introduce intentional data quality issues
    status = random.choices(
        ['PROCESSED', 'PENDING', 'ERROR'],
        weights=[80, 12, 8]
    )[0]
    
    # Some rows will have bad data — null prices, zero quantities
    if random.random() < 0.05:
        price = None  # missing price
    if random.random() < 0.03:
        qty = 0  # zero quantity error

    transactions.append({
        'transaction_id': f'TXN{i+1:04d}',
        'account_id': f'ACC{random.randint(1, NUM_ACCOUNTS):03d}',
        'ticker': random.choice(TICKERS),
        'transaction_type': random.choice(['BUY', 'SELL']),
        'transaction_date': random_date(),
        'quantity': qty,
        'price_per_share': price,
        'total_amount': round(qty * price, 2) if price else None,
        'cost_basis_method': random.choice(METHODS),
        'status': status
    })

df_transactions = pd.DataFrame(transactions)

In [5]:
# Generate cost basis lots
lots = []
for i in range(300):
    qty = random.randint(1, 100)
    cost = round(random.uniform(50, 800), 2)
    
    lots.append({
        'lot_id': f'LOT{i+1:04d}',
        'account_id': f'ACC{random.randint(1, NUM_ACCOUNTS):03d}',
        'ticker': random.choice(TICKERS),
        'acquisition_date': random_date(2019, 2023),
        'quantity': qty,
        'cost_per_share': cost,
        'cost_basis_total': round(qty * cost, 2),
        'is_closed': random.choice([True, False])
    })

df_lots = pd.DataFrame(lots)

In [6]:
# Generate tax reporting
reports = []
for i in range(200):
    proceeds = round(random.uniform(500, 50000), 2)
    cost = round(random.uniform(400, 48000), 2)
    
    # Introduce discrepancies intentionally
    status = random.choices(
        ['VALID', 'DISCREPANCY', 'MISSING_BASIS'],
        weights=[75, 15, 10]
    )[0]

    reports.append({
        'report_id': f'RPT{i+1:04d}',
        'account_id': f'ACC{random.randint(1, NUM_ACCOUNTS):03d}',
        'ticker': random.choice(TICKERS),
        'sale_date': random_date(2022, 2024),
        'proceeds': proceeds,
        'cost_basis': cost if status != 'MISSING_BASIS' else None,
        'gain_loss': round(proceeds - cost, 2) if status != 'MISSING_BASIS' else None,
        'holding_period': random.choice(['SHORT', 'LONG']),
        'reported_to_irs': random.choice([True, False]),
        'status': status
    })

df_reports = pd.DataFrame(reports)

In [7]:
# Save to CSV
df_transactions.to_csv('transactions.csv', index=False)
df_lots.to_csv('cost_basis_lots.csv', index=False)
df_reports.to_csv('tax_reporting.csv', index=False)

print("Data generated successfully")
print(f"Transactions: {len(df_transactions)}")
print(f"Cost basis lots: {len(df_lots)}")
print(f"Tax reports: {len(df_reports)}")

Data generated successfully
Transactions: 500
Cost basis lots: 300
Tax reports: 200
